In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time

### Get md file from SpeedyApply github repo for US based Data Science internships

In [2]:
url = "https://raw.githubusercontent.com/speedyapply/2026-AI-College-Jobs/main/README.md"
response = requests.get(url)

with open('raw_internships.md', 'w', encoding='utf-8') as f:
    f.write(response.text)

### Extract table from raw_internships.md

In [3]:
# Helper function
import re
from io import StringIO

def extract_md_table(md_text, start_marker, end_marker):
    pattern = rf"{re.escape(start_marker)}\n(.*?)\n{re.escape(end_marker)}"
    match = re.search(pattern, md_text, flags=re.DOTALL)

    if not match:
        return None

    table_md = match.group(1).strip()

    df = pd.read_csv(
        StringIO(table_md),
        sep="|",
        engine="python"
    )

    # clean up
    df = df.dropna(axis=1, how="all")
    df.columns = df.columns.str.strip()
    df = df.iloc[1:]  # remove markdown separator row
    df = df.apply(lambda x: x.str.strip())

    # handle company link and name
    if "Company" in df.columns:
        df[["Company Link", "Company"]] = df["Company"].str.extract(
            r'href="([^"]+)".*?<strong>(.*?)</strong>'
        )

    # cleanup position link
    if "Posting" in df.columns:
        df["Posting"] = df["Posting"].str.extract(r'href="([^"]+)"')

    return df


In [4]:
# Load the markdown file
with open("raw_internships.md", "r") as f:
    md_text = f.read()

In [ ]:
# Extract tables into useable dataframes
faang_df = extract_md_table(
    md_text,
    "<!-- TABLE_FAANG_START -->",
    "<!-- TABLE_FAANG_END -->"
)

quant_df = extract_md_table(
    md_text,
    "<!-- TABLE_QUANT_START -->",
    "<!-- TABLE_QUANT_END -->"
)

general_df = extract_md_table(
    md_text,
    "<!-- TABLE_START -->",
    "<!-- TABLE_END -->"
)


In [6]:
faang_df.head()

,Company,Position,Location,Salary,Posting,Age,Company Link
1,Meta,Research Scientist Intern - Mathematics - PhD,"Redmond, WA",$50/hr,https://www.metacareers.com/jobs/1471311707754788,6d,https://about.meta.com
2,Meta,Research Scientist Intern - Optical System Res...,"Redmond, WA",$50/hr,https://www.metacareers.com/jobs/1412438827016753,6d,https://about.meta.com
3,Meta,Research Scientist Intern - Applied Vision - PhD,"Redmond, WA",$50/hr,https://www.metacareers.com/jobs/1112761774275681,6d,https://about.meta.com
4,Meta,Research Scientist Intern - Novel Organic Mate...,"Redmond, WA",$50/hr,https://www.metacareers.com/jobs/884653154471172,6d,https://about.meta.com
5,Meta,Research Scientist Intern - Applied Perception...,"Redmond, WA",$50/hr,https://www.metacareers.com/jobs/2576688855967...,7d,https://about.meta.com


In [7]:
quant_df.head()

,Company,Position,Location,Salary,Posting,Age,Company Link
1,Citadel Securities,Quantitative Research Analyst Intern - BS/MS - US,New York,$125/hr,https://www.citadelsecurities.com/careers/deta...,0d,https://www.citadelsecurities.com/careers
2,Citadel Securities,Machine Learning Researcher - PhD Intern - US,New York,$125/hr,https://www.citadelsecurities.com/careers/deta...,0d,https://www.citadelsecurities.com/careers
3,Citadel Securities,Quantitative Researcher - PhD Intern - US,New York,$125/hr,https://www.citadelsecurities.com/careers/deta...,0d,https://www.citadelsecurities.com/careers
4,Citadel,Machine Learning Researcher - PhD Intern - US,New York,$125/hr,https://www.citadel.com/careers/details/machin...,1d,https://www.citadel.com/careers
5,Citadel,Quantitative Research Analyst Intern - BS/MS - US,New York,$125/hr,https://www.citadel.com/careers/details/quanti...,1d,https://www.citadel.com/careers


In [ ]:
general_df.head()

### Clean the Dataframes

In [ ]:
print("Faang:", faang_df.shape)
print("Quant:", quant_df.shape)
print("General:", general_df.shape)

In [ ]:
general_df.isna().sum()

In [ ]:
general_df.columns

In [ ]:
print("Unique companies in general internships:", general_df.Company.unique().size)

In [13]:
print("Unique companies in faang internships:", faang_df.Company.unique().size)

Unique companies in faang internships: 7


In [14]:
print("Unique companies in quant internships:", quant_df.Company.unique().size)

Unique companies in quant internships: 5


In [ ]:
general_df.Company.value_counts().head(10)

In [ ]:
# I want to visit the links for job postings and extract more info about the interships, especially keywords in job requirement sections.
general_df.Posting.head(10)

In [ ]:
other_10 = general_df[general_df["Company"] == "Tatari"].copy()
other_10

In [18]:
link = other_10["Posting"].head(1)
link = link.values[0]
link

'https://job-boards.greenhouse.io/tatari/jobs/8432059002'

In [19]:
url = "https://job-boards.greenhouse.io/tatari/jobs/8432060002"

headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, "html.parser")

# Main job content container
description = soup.find("div", class_="job__description")

print(description.get_text(separator="\n",strip=True))
str(description)

Tatari is on a mission to revolutionize TV advertising. Founded in 2016 to help transform the antiquated world of TV advertising through the intelligent application of AI and machine learning, Tatari helps some of the world’s fastest growing brands including
Chime, Calm, Tecovas, Manscaped, Saatva, and Liquid I.V
., reach their customers using linear and streaming TV ads. Our platform combines sophisticated media buying with proprietary analytics to turn TV advertising into an automated, digital-like experience, enabling businesses of any size to advertise on TV.
That approach has earned Tatari broad industry recognition, including being named
Best CTV AdTech Platform
in the 8th annual
MarTech Breakthrough Awards
, as well as honors from
Digiday
(
Best Connected TV Platform)
,
AdExchanger
(
Most Innovative TV Advertising Technology
), and
Business Insider
(
Hottest AdTech Companies)
. Tatari has also been recognized as the
Best Place to Work
by
Inc. Magazine
. Backed by an executive te

'<div class="job__description body"><div><p>Tatari is on a mission to revolutionize TV advertising. Founded in 2016 to help transform the antiquated world of TV advertising through the intelligent application of AI and machine learning, Tatari helps some of the world’s fastest growing brands including <em>Chime, Calm, Tecovas, Manscaped, Saatva, and Liquid I.V</em>., reach their customers using linear and streaming TV ads. Our platform combines sophisticated media buying with proprietary analytics to turn TV advertising into an automated, digital-like experience, enabling businesses of any size to advertise on TV.</p>\n<p>That approach has earned Tatari broad industry recognition, including being named\xa0<a href="https://www.tatari.tv/insights/tatari-named-best-ctv-adtech-platform-in-2025-martech-breakthrough-awards">Best CTV AdTech Platform</a> in the 8th annual <em>MarTech Breakthrough Awards</em>, as well as honors from <em>Digiday</em> (<a href="https://www.tatari.tv/insights/tata

In [ ]:
general_df['raw_description'] = None
general_df['description_text'] = None
general_df.head()

In [21]:

# Scraper helper functions — one extractor per job board

import numpy as np
import json
from urllib.parse import urlparse

try:
    from tqdm.notebook import tqdm
except ImportError:
    def tqdm(it, **kw):
        return it

_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

def _get(url, timeout=15):
    return requests.get(url, headers=_HEADERS, timeout=timeout)

def _to_clean(html_str):
    return BeautifulSoup(html_str, "html.parser").get_text(separator="\n", strip=True)

def _try_json_ld(soup):
    """Extract description from JSON-LD JobPosting schema markup."""
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string or "")
            if isinstance(data, list):
                data = data[0] if data else {}
            if data.get("@type") == "JobPosting":
                desc = data.get("description", "")
                if desc:
                    return desc, _to_clean(desc)
        except Exception:
            pass
    return np.nan, np.nan


def _extract_greenhouse(url):
    soup = BeautifulSoup(_get(url).text, "html.parser")
    div = soup.find("div", class_="job__description")
    if div:
        return str(div), div.get_text(separator="\n", strip=True)
    return _try_json_ld(soup)


def _extract_ashby(url):
    """Use Ashby's public posting API."""
    parts = urlparse(url).path.strip("/").split("/")
    if len(parts) >= 2:
        company, job_id = parts[0], parts[1]
        api = f"https://api.ashbyhq.com/posting-api/job-board/{company}/postings/{job_id}"
        data = _get(api).json()
        raw = data.get("descriptionHtml", "")
        if raw:
            return raw, _to_clean(raw)
    return np.nan, np.nan


def _extract_workday(url):
    """Use Workday's internal CXS API, fall back to JSON-LD."""
    parsed = urlparse(url)
    host   = parsed.hostname.split(".")          # e.g. ['caci', 'wd1', 'myworkdayjobs', 'com']
    company, wd = host[0], host[1]
    path   = [p for p in parsed.path.split("/") if p]  # ['en-US', 'board', 'job', ..., 'slug_ID']
    if len(path) >= 2:
        board = path[1]
        m = re.search(r'_([A-Za-z0-9]+)$', path[-1])
        if m:
            job_id = m.group(1)
            api = f"https://{company}.{wd}.myworkdayjobs.com/wday/cxs/{company}/{board}/jobs/{job_id}"
            try:
                data = _get(api).json()
                desc = data.get("jobPostingInfo", {}).get("jobDescription", "")
                if desc:
                    return desc, _to_clean(desc)
            except Exception:
                pass
    # Fallback: JSON-LD on the rendered page
    try:
        soup = BeautifulSoup(_get(url).text, "html.parser")
        return _try_json_ld(soup)
    except Exception:
        return np.nan, np.nan


def _extract_lever(url):
    clean_url = re.sub(r'/apply/?$', '', url)   # strip trailing /apply
    soup = BeautifulSoup(_get(clean_url).text, "html.parser")
    div = (
        soup.find("div", {"data-qa": "job-description"})
        or soup.find("div", class_="section-wrapper")
        or soup.find("div", class_="content")
    )
    if div:
        return str(div), div.get_text(separator="\n", strip=True)
    return _try_json_ld(soup)


def _extract_smartrecruiters(url):
    """Use SmartRecruiters public API, fall back to page scrape."""
    parsed = urlparse(url)
    parts  = parsed.path.strip("/").split("/")   # [CompanyName, jobId-slug...]
    if len(parts) >= 2:
        company = parts[0]
        job_id  = parts[1].split("-")[0]
        api = f"https://api.smartrecruiters.com/v1/companies/{company}/postings/{job_id}"
        try:
            data = _get(api).json()
            sections = data.get("jobAd", {}).get("sections", {})
            texts = [
                sections.get(k, {}).get("text", "")
                for k in ("jobDescription", "qualifications", "additionalInformation")
            ]
            combined = "\n\n".join(t for t in texts if t)
            if combined:
                return combined, _to_clean(combined)
        except Exception:
            pass
    try:
        soup = BeautifulSoup(_get(url).text, "html.parser")
        div = soup.find("div", class_=re.compile(r"job-description", re.I))
        if div:
            return str(div), div.get_text(separator="\n", strip=True)
        return _try_json_ld(soup)
    except Exception:
        return np.nan, np.nan


def _extract_generic(url):
    soup = BeautifulSoup(_get(url).text, "html.parser")
    result = _try_json_ld(soup)
    if pd.notna(result[0]):
        return result
    for tag, attrs in [
        ("div", {"id": "job-description"}),
        ("div", {"class": "job-description"}),
        ("div", {"class": "jobDescription"}),
        ("div", {"class": "description"}),
        ("div", {"class": "posting-description"}),
        ("section", {"class": re.compile(r"description", re.I)}),
    ]:
        div = soup.find(tag, attrs)
        if div:
            return str(div), div.get_text(separator="\n", strip=True)
    return np.nan, np.nan


def fetch_description(url):
    """Route to the right extractor. Returns (raw_html, clean_text) or (NaN, NaN)."""
    if not url or (isinstance(url, float) and np.isnan(url)):
        return np.nan, np.nan
    try:
        domain = urlparse(url).hostname or ""
        if "greenhouse.io" in domain:
            raw, text = _extract_greenhouse(url)
        elif "ashbyhq.com" in domain:
            raw, text = _extract_ashby(url)
        elif "myworkdayjobs.com" in domain:
            raw, text = _extract_workday(url)
        elif "lever.co" in domain:
            raw, text = _extract_lever(url)
        elif "smartrecruiters.com" in domain:
            raw, text = _extract_smartrecruiters(url)
        else:
            raw, text = _extract_generic(url)
        return (
            raw  if pd.notna(raw)  else np.nan,
            text if pd.notna(text) else np.nan,
        )
    except Exception:
        return np.nan, np.nan


In [ ]:

# Scrape job descriptions for all rows in general_df

DELAY = 0.5  # seconds between requests

raw_descs, clean_descs = [], []

for i, url in enumerate(tqdm(general_df["Posting"], total=len(general_df), desc="Fetching descriptions")):
    raw, clean = fetch_description(url)
    raw_descs.append(raw)
    clean_descs.append(clean)

    if (i + 1) % 50 == 0:
        n_ok = sum(1 for x in clean_descs if pd.notna(x))
        print(f"  [{i+1:3d}/{len(general_df)}] extracted so far: {n_ok}")

    time.sleep(DELAY)

general_df["raw_description"]  = raw_descs
general_df["description_text"] = clean_descs

n_ok = general_df["description_text"].notna().sum()
print(f"\nDone! Extracted {n_ok}/{len(general_df)} descriptions ({100*n_ok/len(general_df):.1f}%)")
general_df[["Company", "Position", "description_text"]].head(10)


In [ ]:
general_df.isna().sum()

In [ ]:
general_df.shape

In [ ]:
general_df["description_text"][10]